In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
!{sys.executable} -m pip install -e .



error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [16]:
import sys
print(sys.executable)
print(sys.version)


/Users/davidguyomarch/Projects/SophieJanitor/rag-env-312/bin/python
3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]


In [5]:
import sys
import os

# Trouve la racine du projet (là où se trouve le dossier src)
current_dir = os.getcwd()
if 'notebooks' in current_dir:
    # Si on est dans notebooks/, remonte d'un niveau
    project_root = os.path.dirname(current_dir)
else:
    # Sinon on est déjà à la racine
    project_root = current_dir

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Working directory: {current_dir}")
print(f"Adding to path: {src_path}")

from sophie_janitor import SophieJanitor

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("sys.path:", sys.path)


Working directory: /Users/davidguyomarch/Projects/SophieJanitor
Adding to path: /Users/davidguyomarch/Projects/SophieJanitor/src
Python executable: /Users/davidguyomarch/Projects/SophieJanitor/rag-env-312/bin/python
Python version: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]
sys.path: ['/Users/davidguyomarch/Projects/SophieJanitor/src', '/opt/homebrew/Cellar/python@3.12/3.12.12_2/Frameworks/Python.framework/Versions/3.12/lib/python312.zip', '/opt/homebrew/Cellar/python@3.12/3.12.12_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12', '/opt/homebrew/Cellar/python@3.12/3.12.12_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/lib-dynload', '', '/Users/davidguyomarch/Projects/SophieJanitor/rag-env-312/lib/python3.12/site-packages']


In [6]:
packages = [
    "chromadb",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-chroma",
    "langchain-ollama",
]

import importlib.metadata

for p in packages:
    try:
        print(f"{p}:", importlib.metadata.version(p))
    except:
        print(f"{p}: not installed")


chromadb: 1.5.0
langchain: 1.2.10
langchain-core: 1.2.12
langchain-community: 0.4.1
langchain-chroma: 1.1.0
langchain-ollama: 1.0.1


# Test complet 

In [35]:
from sophie_janitor import SophieJanitor

sj = SophieJanitor()

result = sj.ask(
    "Quelles sont les conditions de la légitime défense ?",
    threshold=1.0,
    distance_mode=True
)

print(result["answer"])
print("Articles cités :", result["articles_cites"])

 Les conditions de la légitime défense sont définies dans les articles 122-5 et 122-6 du Code pénal. Selon ces dispositions, une personne est présumée avoir agi en état de légitime défense lorsqu'elle accomplit un acte pour repousser l'entrée par effraction, violence ou ruse dans un lieu habité (article 122-6, alinéa 1°) ou pour se défendre contre les auteurs de vols ou de pillages exécutés avec violence (article 122-6, alinéa 2°). De plus, une personne ne peut être pénalement responsable si elle accomplit un acte commandé par la nécessité de la légitime défense d'elle-même ou d'autrui, sauf s'il y a disproportion entre les moyens de défense employés et la gravité de l'atteinte (article 122-5). Enfin, une personne ne peut être pénalement responsable pour avoir interrompu l'exécution d'un crime ou d'un délit contre un bien, lorsque cet acte est strictement nécessaire au but poursuivi et que les moyens employés sont proportionnés à la gravité de l'infraction (article 122-5).
Articles cit

# Construction de la base de données des articles du code pénal

In [34]:
from indexing import Indexer
from langchain_ollama import OllamaEmbeddings
from sophie_janitor import CodePenalIngestor

# 1️⃣ Parsing
ingestor = CodePenalIngestor("./data/LEGITEXT000006070719.pdf")
documents = ingestor.parse()
print(f"Nb articles trouvés: {len(documents)}")

# 2️⃣ Indexation
embeddings = OllamaEmbeddings(model="bge-m3")

indexer = Indexer(
    embeddings=embeddings,
    persist_directory="./chroma_code_penal",
    collection_name="code_penal"
)

indexer.build(documents)

Nb articles trouvés: 2440


In [29]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3")

vectorstore = Chroma(
    persist_directory="./chroma_code_penal",
    embedding_function=embeddings,
    collection_name="code_penal",
)

# Accès brut à la collection
collection = vectorstore._collection

data = collection.get(limit=5)

print("IDS:", data["ids"])
print("METADATAS:", data["metadatas"])
#print("DOCUMENTS:", data["documents"][0])


IDS: []
METADATAS: []


In [ ]:
docs = CodePenalIngestor("./data/LEGITEXT000006070719.pdf").parse()
print(type(docs))
print(len(docs))
print(docs[:3])

<class 'list'>
2440
[Document(metadata={'article': '111-1', 'code': 'Code pénal', 'type': 'article'}, page_content='Les infractions pénales sont classées, suivant leur gravité, en crimes, délits et contraventions.'), Document(metadata={'article': '111-2', 'code': 'Code pénal', 'type': 'article'}, page_content='La loi détermine les crimes et délits et fixe les peines applicables à leurs auteurs. Le règlement détermine les contraventions et fixe, dans les limites et selon les distinctions établies par la loi, les peines applicables aux contrevenants.'), Document(metadata={'article': '111-3', 'code': 'Code pénal', 'type': 'article'}, page_content="Nul ne peut être puni pour un crime ou pour un délit dont les éléments ne sont pas définis par la loi, ou pour une contravention dont les éléments ne sont pas définis par le règlement. Nul ne peut être puni d'une peine qui n'est pas prévue par la loi, si l'infraction est un crime ou un délit, ou par le règlement, si l'infraction est une contrave